# Introduction

In [1]:
import pandas as pd
import numpy as np
import zipfile
import seaborn as sns
import matplotlib.pyplot as plt

# Web-Scrapping

I can have everything on Basketball-reference, so I will get everything since 2000 from the website.

In [2]:
from bs4 import BeautifulSoup, Comment
import requests
import time

## Team data

In [4]:
#Put all the criteria in a list
stats = {
    'school' : 'Avg',
    'opponent' : 'Opps',
    'advanced' : 'Adv',
    'advanced-opponent' : 'Adv-Opps',
    }

In [ ]:
#Create a function to optimize the code
def get_player(url, exclude_pos, dict_of_pos):
    try:
        response = requests.get(url)
        html_content = response.content
        soup = BeautifulSoup(html_content, 'html.parser')
        tables = soup.find_all('table', id=lambda x: x not in exclude_pos)
        for i, table in enumerate(tables):
            if i in exclude_pos:
                continue  
            else:
                #headers
                headers = []
                header_row = table.find('thead').find('tr') if table.find('thead') else table.find('tr')
                for th in header_row.find_all('th'):
                    headers.append(th.text.strip())
                #rows
                rows = []
                for row in table.find('tbody').find_all('tr') if table.find('tbody') else table.find_all('tr'):
                    cells = row.find_all(['td', 'th'])
                    if len(cells) > 0:  #Ensure the row contains data
                        rows.append([cell.text.strip() for cell in cells])
                #merge on basis of players' name
                if i == 0:
                    df = pd.DataFrame(rows, columns=headers)
                else:
                    granulated = pd.DataFrame(rows, columns=headers)
                    granulated = granulated.add_prefix(f'{dic_of_position[i]}_')
                    granulated = granulated.rename(columns={f'{dic_of_position[i]}_Player' : 'Player'})
                    yearly = pd.merge(yearly, granulated, on="Player")
    except:
        print(url)

    return df

In [ ]:
dico_of_error = {}
exclude_pos = [1, 2, 3] #tables to exclude
dic_of_position = { #different tables
    0 : "player_info",
    3 : "average",
    4 : "totals",
    5 : "per_40",
    6 : "per_100",
    7 : "advanced"
}

headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36",
    "Referer": "https://www.sports-reference.com/",
    "Accept-Language": "en-US,en;q=0.9",
}

session = requests.Session()
session.headers.update(headers)

for i in range(1, 25):
    if i < 10:
        year = f'200{str(i)}'
    else:
        year = f'20{str(i)}'
    for key, value in stats.items():
        print(key, year)
        #Get the HTML Content
        url = f'https://www.sports-reference.com/cbb/seasons/men/{year}-{key}-stats.html'
        response = requests.get(url)

       #Check if the request was successful
        if response.status_code == 200:
            html_content = response.content
        else:
            print(f"Failed to retrieve the page. Status code: {response.status_code}")
            exit()
        #Parse the HTML content 
        soup = BeautifulSoup(html_content, 'html.parser')
        table = soup.find('table') 
        #Get the table header
        headers = ["School",	"G", "W", "L",	"W-L%",	"SRS",	"SOS", "del1",
                "W",	"L", "del2", "W", "L", "del3", "W", "L", "del4", "Tm.", "Opp.",
                    "del5", "MP", "FG",	"FGA", "FG%", "3P",	"3PA",	"3P%", "FT",
                    "FTA",	"FT%",	"ORB",	"TRB",	"AST",	"STL",	"BLK",	"TOV",	"PF"]
        #Get the table rows
        rows = []
        for row in table.find('tbody').find_all('tr') if table.find('tbody') else table.find_all('tr'):
            cells = row.find_all('td')
            if len(cells) > 0:  #Ensure the row contains data
                rows.append([cell.text.strip() for cell in cells])

        #Convert the data into a pandas DataFrame
        season_college_team = pd.DataFrame(rows, columns=headers)
        
        #Go into each schools in a season and gather the players data
        for j, school in enumerate(season_college_team["School"].str.lower().str.replace(' ', '-')):
            try:
                teamly = season_college_team["School"].str.lower().str.replace(' ', '-').apply(
                    lambda row: get_player(url=f'https://www.sports-reference.com/cbb/schools/{row}/men/{year}.html',
                                           exclude_pos=exclude_pos, dic_of_position=dic_of_position)
                )
                if j == 0: #Create a new aggregate db if it is the first iteration
                    yearly = teamly.copy()
                else: #Add on the db if it is more than the first iteration
                    yearly = pd.concat([yearly, teamly])
            except: #If the team is wrongly written, put in the error dict with the year 
                dico_of_error[school] = year
            
        season_college_team.to_csv(f'Team NCAA {year} {key}.csv', index=False) #Team data
        yearly.to_csv(f'Player NCAA {year}.csv', index=False)

In [ ]:
##Configuration
#Put all the criteria in a list
years = [2006]
stats = {
    'school' : 'Avg',
    'opponent' : 'Opps',
    'advanced' : 'Adv',
    'advanced-opponent' : 'Adv-Opps',
    }
BASE_URL = "https://www.sports-reference.com"
REQUEST_DELAY = 3.3 #Maximum of 20 requests per minutes
dico_of_error = {} #Dico with Team-Year as Key-Value in case HTML page not found
exclude_pos_1 = [1, 2, 3, 4] #tables to exclude if len == 9
exclude_pos_2 = [ #Tables to exclude if len > 9
    1, 2, 3, 4, 6, 8, 
    9, 10, 12, 14
]
dic_of_position_1 = { #different tables
    0 : "info",
    5 : "average",
    6 : "totals",
    7 : "per_100",
    8 : "advanced"
    }
dic_of_position_2 = { #different tables
    0 : "info",
    5 : "average",
    7 : "totals",
    11 : "per_100",
    13 : "advanced"
    }
HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36",
    "Referer": BASE_URL,
    "Accept-Language": "en-US,en;q=0.9",
    }
session = requests.Session()
session.headers.update(HEADERS)
schools = ['ucla', 'air-force']
for year in years:
    for j, school in enumerate(schools):
        time.sleep(REQUEST_DELAY)
        player_url = BASE_URL + f'/cbb/schools/{school}/men/{year}.html'  #Replace with the URL of the site containing the table
        player_html = fetch_html(player_url, session) #Get the HTML content
        soup = BeautifulSoup(player_html, 'html.parser')
        tables = soup.find_all('table')
        comments = soup.find_all(string=lambda text: isinstance(text, Comment))
        for comment in comments:
            comment_soup = BeautifulSoup(comment, 'html.parser')
            comment_tables = comment_soup.find_all('table')
            for table in comment_tables:
                tables.append(table)
        if len(tables) == 9:
            exclude_pos = exclude_pos_1
            dic_of_position = dic_of_position_1
        else:
            exclude_pos = exclude_pos_2
            dic_of_position = dic_of_position_2
        for i, table in enumerate(tables):
            if i in exclude_pos:
                continue  #Skip excluded tables
            else:
                #headers
                headers = []
                header_row = table.find('thead').find('tr') if table.find('thead') else table.find('tr')
                for th in header_row.find_all('th'):
                    headers.append(th.text.strip())
                #rows
                rows = []
                for row in table.find('tbody').find_all('tr') if table.find('tbody') else table.find_all('tr'):
                    cells = row.find_all(['td', 'th'])
                    if len(cells) > 0:  # Ensure the row contains data
                        rows.append([cell.text.strip() for cell in cells])

                    #merge on basis of players' name
                if i == 0: #if it is the first table from the HTML pages
                    teamly = pd.DataFrame(rows, columns=headers)
                    teamly['Team'] = school
                    print(school)
                else:
                    granulated = pd.DataFrame(rows, columns=headers)
                    granulated = granulated.add_prefix(f'{dic_of_position[i]}_')
                    granulated = granulated.rename(columns={f'{dic_of_position[i]}_Player' : 'Player'})
                    teamly = teamly.rename(columns={f'{dic_of_position[i]}_Player' : 'Player'})
                    teamly = pd.merge(teamly, granulated, on="Player", how='left')
                    
                if j == 0: #Create a new aggregated db if it is the first iteration
                    yearly = teamly.copy()
                else: #Add on the db if it is more than the first iteration
                    if i == len(tables)-1: #When all the tables have been merged to the db 
                        yearly = pd.concat([yearly, teamly], ignore_index = True)
                granulated = pd.DataFrame()
                

ucla
air-force


In [20]:
yearly

,Player,#,Class,Pos,Height,Weight,Hometown,RSCI Top 100,Summary,Team,...,per_100_TRB%,per_100_AST%,per_100_BLK%,per_100_TOV%,per_100_USG%,per_100_OWS,per_100_DWS,per_100_WS,per_100_WS/40,per_100_Awards
0,Arron Afflalo,4,SO,G,6-5,215,"Compton, CA",26 (2004),"15.8 Pts, 4.2 Reb, 1.8 Ast",ucla,...,8.1,11.9,0.3,13.3,24.5,4.4,1.7,6.1,.187,
1,Jordan Farmar,1,SO,G,6-2,180,,20 (2004),"13.5 Pts, 2.6 Reb, 5.1 Ast",ucla,...,5.4,37.6,0.6,21.9,29.4,1.7,2.2,3.9,.139,
2,Luc Richard Mbah a Moute,23,FR,F,6-8,230,,,"9.1 Pts, 8.2 Reb, 1.3 Ast",ucla,...,17.8,9.3,1.5,17.7,17.1,2.3,3.5,5.8,.202,
3,Cedric Bozeman,21,SR,G,6-6,207,,19 (2001),"7.6 Pts, 3.3 Reb, 2.3 Ast",ucla,...,7.7,16.7,0.2,21.6,15.8,1.7,1.3,3.0,.143,
4,Ryan Hollins,15,SR,C,7-0,230,,,"7.0 Pts, 4.8 Reb, 0.3 Ast",ucla,...,14.2,2.9,3.2,17.0,16.9,1.8,2.1,3.9,.220,
5,Darren Collison,2,FR,G,6-1,160,"Rancho Cucamonga, CA",98 (2005),"5.5 Pts, 1.8 Reb, 2.3 Ast",ucla,...,6.1,23.7,0.2,26.1,21.2,0.4,1.4,1.8,.096,
6,Michael Roll,20,FR,G,6-5,200,"Aliso Viejo, CA",,"3.4 Pts, 0.9 Reb, 0.9 Ast",ucla,...,4.0,12.4,0.3,17.2,14.6,0.5,0.6,1.1,.078,
7,Alfred Aboya,12,FR,F,6-9,245,,81 (2005),"3.6 Pts, 2.4 Reb, 0.4 Ast",ucla,...,11.0,5.2,1.2,17.6,13.8,0.9,1.0,1.9,.161,
8,Lorenzo Mata-Real,14,SO,C,6-9,237,"Huntington Park, CA",,"3.6 Pts, 3.9 Reb, 0.0 Ast",ucla,...,18.0,0.7,6.4,17.1,16.1,0.3,1.5,1.7,.238,
9,Ryan Wright,1,FR,F,6-9,238,,46 (2005),"2.4 Pts, 1.5 Reb, 0.0 Ast",ucla,...,9.7,0.7,1.0,21.9,15.3,0.3,0.5,0.8,.105,


In [4]:
#Function to get HTML content from an URL
def fetch_html(url, session):
    try:
        response = session.get(url)
        if response.status_code == 200:
            return response.content
        else:
            print(f"Échec de la requête : {response.status_code} pour {url}")
            return None
    except Exception as e:
        print(f"Erreur lors de la récupération de {url} : {e}")
        return None

In [7]:
##Configuration
#Put all the criteria in a list
stats = {
    'school' : 'Avg',
    'opponent' : 'Opps',
    'advanced' : 'Adv',
    'advanced-opponent' : 'Adv-Opps',
    }
BASE_URL = "https://www.sports-reference.com"
REQUEST_DELAY = 3.05 #Maximum of 20 requests per minutes
dico_of_error = {} #Dico with Team-Year as Key-Value in case HTML page not found
exclude_pos_1 = [1, 2, 3, 4] #tables to exclude if len == 9
exclude_pos_2 = [ #Tables to exclude if len > 9
    1, 2, 3, 4, 6, 8, 
    9, 10, 12, 14
]
dic_of_position_1 = { #different tables
    0 : "info",
    5 : "average",
    6 : "totals",
    7 : "per_100",
    8 : "advanced"
    }
dic_of_position_2 = { #different tables
    0 : "info",
    5 : "average",
    7 : "totals",
    11 : "per_100",
    13 : "advanced"
    }
HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36",
    "Referer": BASE_URL,
    "Accept-Language": "en-US,en;q=0.9",
    }
session = requests.Session()
session.headers.update(HEADERS)

In [ ]:
##Scrapping
for key, value in stats.items():
    for year in range(2001, 2002):
        time.sleep(REQUEST_DELAY)
        #Get the HTML Content
        team_url = BASE_URL + f'/cbb/seasons/men/{str(year)}-{key}-stats.html'  
        try:
            team_html = fetch_html(team_url, session) #Get the HTML content

            #Parse the HTML content 
            soup = BeautifulSoup(team_html, 'html.parser')
            table = soup.find('table')  #Get the table with the school stats

            #Get the table header - Already precompleted because of errors
            headers = ["School",	"G", "W", "L",	"W-L%",	"SRS",	"SOS", "del1",
                    "W",	"L", "del2", "W", "L", "del3", "W", "L", "del4", "Tm.", "Opp.",
                        "del5", "MP", "FG",	"FGA", "FG%", "3P",	"3PA",	"3P%", "FT",
                        "FTA",	"FT%",	"ORB",	"TRB",	"AST",	"STL",	"BLK",	"TOV",	"PF"]
            #Get the table rows
            rows = []
            for row in table.find('tbody').find_all('tr') if table.find('tbody') else table.find_all('tr'):
                cells = row.find_all('td')
                if len(cells) > 0:  #Ensure the row contains data
                    rows.append([cell.text.strip() for cell in cells])

            #Convert the Teams data into a pandas DataFrame
            season_college_team = pd.DataFrame(rows, columns=headers)
            season_college_team.to_csv(f'Team NCAA {str(year)} {key}.csv', index=False) #Team data
        except:
            continue
        if key == "school":
            #Apply text treatment
            season_college_team['school_clean'] = season_college_team['School'].str.replace(r"[()&.']", "", regex=True)  # Remove "(", ")", ".", "&", "'", and spaces
            season_college_team['school_clean'] = season_college_team['school_clean'].str.replace(r"\s+", "-", regex=True)  # Replace remaining spaces with "-"
            season_college_team['school_clean'] = season_college_team['school_clean'].str.replace("-NCAA", "", regex=True)
            season_college_team['school_clean'] = season_college_team['school_clean'].str.strip()  # Remove leading and trailing spaces
            #Go into each schools in a season and gather the players data
            for j, school in enumerate(season_college_team["school_clean"].str.lower()):
                time.sleep(REQUEST_DELAY)
                print(f'{year} :', round(j/len(season_college_team), 3)*100, "%")
                try: #Try in case of a typo in the college name 
                    #Get the HTML Content
                    player_url = BASE_URL + f'/cbb/schools/{school}/men/{year}.html'
                    player_html = fetch_html(player_url, session) #Get the HTML content
                    soup = BeautifulSoup(player_html, 'html.parser')
                    tables = soup.find_all('table')
                    comments = soup.find_all(string=lambda text: isinstance(text, Comment))
                    for comment in comments:
                        comment_soup = BeautifulSoup(comment, 'html.parser')
                        comment_tables = comment_soup.find_all('table')
                        for table in comment_tables:
                            #Process tables found in comments
                            tables.append(table)
                            if len(tables) == 9:
                                exclude_pos = exclude_pos_1
                                dic_of_position = dic_of_position_1
                            else:
                                exclude_pos = exclude_pos_2
                                dic_of_position = dic_of_position_2
                            for i, table in enumerate(tables):
                                if i in exclude_pos:
                                    continue  #Skip excluded tables
                                else:
                                    #headers
                                    headers = []
                                    header_row = table.find('thead').find('tr') if table.find('thead') else table.find('tr')
                                    for th in header_row.find_all('th'):
                                        headers.append(th.text.strip())
                                    #rows
                                    rows = []
                                    for row in table.find('tbody').find_all('tr') if table.find('tbody') else table.find_all('tr'):
                                        cells = row.find_all(['td', 'th'])
                                        if len(cells) > 0:  #Ensure the row contains data
                                            rows.append([cell.text.strip() for cell in cells])

                                        #merge on basis of players' name
                                    if i == 0: #if it is the first table from the HTML pages
                                        teamly = pd.DataFrame(rows, columns=headers)
                                        teamly['Team'] = school
                                    else:
                                        granulated = pd.DataFrame(rows, columns=headers)
                                        granulated = granulated.add_prefix(f'{dic_of_position[i]}_')
                                        granulated = granulated.rename(columns={f'{dic_of_position[i]}_Player' : 'Player'})
                                        teamly = teamly.rename(columns={f'{dic_of_position[i]}_Player' : 'Player'})
                                        teamly = pd.merge(teamly, granulated, on=["Player"], how='left')
                                        
                                    if j == 0: #Create a new aggregated db if it is the first iteration
                                        yearly = teamly.copy()
                                    else: #Add on the db if it is more than the first iteration
                                        if i == len(tables)-1: #When all the tables have been merged to the db 
                                            yearly = pd.concat([yearly, teamly], ignore_index = True)
                                    granulated = pd.DataFrame()
                except: #If the team is wrongly written, put in the error dict with the year 
                    dico_of_error[school] = str(year)

            #Save the DataFrame to a CSV file
            yearly.to_csv(f'Player NCAA {str(year)}.csv', index=False) #Player data

2001 : 0.0 %
2001 : 0.3 %
2001 : 0.6 %
2001 : 0.8999999999999999 %
2001 : 1.3 %
2001 : 1.6 %
2001 : 1.9 %
2001 : 2.1999999999999997 %
2001 : 2.5 %
2001 : 2.8000000000000003 %
2001 : 3.1 %
2001 : 3.5000000000000004 %
2001 : 3.8 %
2001 : 4.1000000000000005 %
2001 : 4.3999999999999995 %
2001 : 4.7 %
2001 : 5.0 %
2001 : 5.3 %
2001 : 5.7 %
2001 : 6.0 %
2001 : 6.3 %
2001 : 6.6000000000000005 %
2001 : 6.9 %
2001 : 7.199999999999999 %
2001 : 7.5 %
2001 : 7.9 %
2001 : 8.200000000000001 %
2001 : 8.5 %
2001 : 8.799999999999999 %
2001 : 9.1 %
2001 : 9.4 %
2001 : 9.700000000000001 %
2001 : 10.100000000000001 %
2001 : 10.4 %
2001 : 10.7 %
2001 : 11.0 %
2001 : 11.3 %
2001 : 11.600000000000001 %
2001 : 11.899999999999999 %
2001 : 12.3 %
2001 : 12.6 %
2001 : 12.9 %
2001 : 13.200000000000001 %
2001 : 13.5 %
2001 : 13.8 %
2001 : 14.2 %
2001 : 14.499999999999998 %
2001 : 14.799999999999999 %
2001 : 15.1 %
2001 : 15.4 %
2001 : 15.7 %
2001 : 16.0 %
2001 : 16.400000000000002 %
2001 : 16.7 %
2001 : 17.0 %
200

In [16]:
yearly[yearly['Player'] == 'Craig Lewis']

,Player,Class,Pos,Height,RSCI Top 100,Summary,Team


In [ ]:
##Scrapping
for key, value in stats.items():
    for year in range(2001, 2002):
        #Get the HTML Content
        team_url = BASE_URL + f'/cbb/seasons/men/{str(year)}-{key}-stats.html'
        try:
            team_html = fetch_html(team_url, session) #Get the HTML content

            #Parse the HTML content 
            soup = BeautifulSoup(team_html, 'html.parser')
            table = soup.find('table')  #Get the table with the school stats

            #Get the table header - Already precompleted because of errors
            headers = ["School",	"G", "W", "L",	"W-L%",	"SRS",	"SOS", "del1",
                    "W",	"L", "del2", "W", "L", "del3", "W", "L", "del4", "Tm.", "Opp.",
                        "del5", "MP", "FG",	"FGA", "FG%", "3P",	"3PA",	"3P%", "FT",
                        "FTA",	"FT%",	"ORB",	"TRB",	"AST",	"STL",	"BLK",	"TOV",	"PF"]
            #Get the table rows
            rows = []
            for row in table.find('tbody').find_all('tr') if table.find('tbody') else table.find_all('tr'):
                cells = row.find_all('td')
                if len(cells) > 0:  #Ensure the row contains data
                    rows.append([cell.text.strip() for cell in cells])

            #Convert the Teams data into a pandas DataFrame
            season_college_team = pd.DataFrame(rows, columns=headers)
            season_college_team.to_csv(f'Team NCAA {str(year)} {key}.csv', index=False) #Team data
        except:
            continue
        if key == "school":
            # Apply text treatment
            season_college_team['school_clean'] = season_college_team['School'].str.replace(r"[()&.']", "", regex=True)  #Remove "(", ")", ".", "&", "'", and spaces
            season_college_team['school_clean'] = season_college_team['school_clean'].str.replace(r"\s+", "-", regex=True)  #Replace remaining spaces with "-"
            season_college_team['school_clean'] = season_college_team['school_clean'].str.replace("-NCAA", "", regex=True)
            season_college_team['school_clean'] = season_college_team['school_clean'].str.strip()  #Remove leading and trailing spaces
            #Go into each schools in a season and gather the players data
            for j, school in enumerate(season_college_team["school_clean"].str.lower()):
                try: #Try in case of a typo in the college name 
                    #Get the HTML Content
                    player_url = BASE_URL + f'/cbb/schools/{school}/men/{str(year)}.html'
                    player_html = fetch_html(player_url, session) #Get the HTML content
                    soup = BeautifulSoup(player_html, 'html.parser')
                    tables = soup.find_all('table', id=lambda x: x not in exclude_pos)
                    for i, table in enumerate(tables):
                        if i in exclude_pos:
                            continue  #Skip excluded tables
                        else:
                            #headers
                            headers = []
                            header_row = table.find('thead').find('tr') if table.find('thead') else table.find('tr')
                            for th in header_row.find_all('th'):
                                headers.append(th.text.strip())
                            #rows
                            rows = []
                            for row in table.find('tbody').find_all('tr') if table.find('tbody') else table.find_all('tr'):
                                cells = row.find_all(['td', 'th'])
                                if len(cells) > 0:  #Ensure the row contains data
                                    rows.append([cell.text.strip() for cell in cells])

                            #merge on basis of players' name
                            if i == 0:
                                teamly = pd.DataFrame(rows, columns=headers)
                                teamly['Team'] = school
                            else:
                                granulated = pd.DataFrame(rows, columns=headers)
                                granulated = granulated.add_prefix(f'{dic_of_position[i]}_')
                                granulated = granulated.rename(columns={f'{dic_of_position[i]}_Player' : 'Player'})
                                teamly = pd.merge(yearly, granulated, on="Player")
                    if j == 0: #Create a new aggregated db if it is the first iteration
                        yearly = teamly.copy()
                    else: #Add on the db if it is more than the first iteration
                        yearly = pd.concat([yearly, teamly])
                except: #If the team is wrongly written, put in the error dict with the year 
                    dico_of_error[school] = str(year)
                
            yearly.to_csv(f'Player NCAA {str(year)}.csv', index=False) #Player data

In [ ]:
#Config
BASE_URL = "https://www.sports-reference.com"
HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36",
    "Referer": BASE_URL,
    "Accept-Language": "en-US,en;q=0.9",
}
REQUEST_DELAY = 6  #10 requests per minute
EXCLUDE_POS = [1, 2, 3]  #To exclude
DIC_OF_POSITION = {  #All the tables
    0: "player_info",
    3: "average",
    4: "totals",
    5: "per_40",
    6: "per_100",
    7: "advanced"
}
#Dictionnary to store the erros
dico_of_error = {}


In [ ]:

def fetch_html(url, session):
    try:
        response = session.get(url)
        if response.status_code == 200:
            return response.content
        else:
            print(f"Échec de la requête : {response.status_code} pour {url}")
            return None
    except Exception as e:
        print(f"Erreur lors de la récupération de {url} : {e}")
        return None

def parse_table(table):
    headers = []
    header_row = table.find('thead').find('tr') if table.find('thead') else table.find('tr')
    for th in header_row.find_all('th'):
        headers.append(th.text.strip())
    
    rows = []
    for row in table.find('tbody').find_all('tr') if table.find('tbody') else table.find_all('tr'):
        cells = row.find_all(['td', 'th'])
        if len(cells) > 0:
            rows.append([cell.text.strip() for cell in cells])
    
    return pd.DataFrame(rows, columns=headers)

# Fonction principale
def main():
    session = requests.Session()
    session.headers.update(HEADERS)

    for year in range(2001, 2025):  # De 2001 à 2024
        year_str = f"20{str(year).zfill(2)}" if year >= 2010 else f"200{str(year)}"
        print(f"Traitement de l'année {year_str}...")

        # Récupérer les données de l'équipe
        team_url = f"{BASE_URL}/cbb/seasons/men/{year_str}-school-stats.html"
        team_html = fetch_html(team_url, session)
        if not team_html:
            continue

        soup = BeautifulSoup(team_html, 'html.parser')
        table = soup.find('table')
        if not table:
            print(f"Aucune table trouvée pour l'année {year_str}.")
            continue

        # Convertir la table en DataFrame
        team_headers = ["School", "G", "W", "L", "W-L%", "SRS", "SOS", "del1",
                        "W", "L", "del2", "W", "L", "del3", "W", "L", "del4", "Tm.", "Opp.",
                        "del5", "MP", "FG", "FGA", "FG%", "3P", "3PA", "3P%", "FT",
                        "FTA", "FT%", "ORB", "TRB", "AST", "STL", "BLK", "TOV", "PF"]
        team_rows = []
        for row in table.find('tbody').find_all('tr') if table.find('tbody') else table.find_all('tr'):
            cells = row.find_all('td')
            if len(cells) > 0:
                team_rows.append([cell.text.strip() for cell in cells])
        
        season_college_team = pd.DataFrame(team_rows, columns=team_headers)
        season_college_team.to_csv(f'Team NCAA {year_str}.csv', index=False)

        # Récupérer les données des joueurs pour chaque école
        yearly_data = None
        for school in season_college_team["School"].str.lower().str.replace(' ', '-'):
            try:
                school_url = f"{BASE_URL}/cbb/schools/{school}/men/{year_str}.html"
                school_html = fetch_html(school_url, session)
                if not school_html:
                    continue

                soup = BeautifulSoup(school_html, 'html.parser')
                tables = soup.find_all('table')
                teamly_data = None

                for i, table in enumerate(tables):
                    if i in EXCLUDE_POS:
                        continue
                    table_data = parse_table(table)
                    if i == 0:
                        teamly_data = table_data
                    else:
                        table_data = table_data.add_prefix(f'{DIC_OF_POSITION[i]}_')
                        table_data = table_data.rename(columns={f'{DIC_OF_POSITION[i]}_Player': 'Player'})
                        teamly_data = pd.merge(teamly_data, table_data, on="Player", how="outer")

                if yearly_data is None:
                    yearly_data = teamly_data
                else:
                    yearly_data = pd.concat([yearly_data, teamly_data])

                time.sleep(REQUEST_DELAY)  # Respecter le délai entre les requêtes
            except Exception as e:
                print(f"Erreur pour l'école {school} en {year_str} : {e}")
                dico_of_error[school] = year_str

        if yearly_data is not None:
            yearly_data.to_csv(f'Player NCAA {year_str}.csv', index=False)

        print(f"Données sauvegardées pour l'année {year_str}.")

    print("Traitement terminé.")
    print(f"Erreurs rencontrées : {dico_of_error}")

if __name__ == "__main__":
    main()

## Regular Season extraction

Scrapping data from basketball-realgm from all the season (Both March Madness and regular season's games).

In [2]:
from bs4 import BeautifulSoup
import requests

In [ ]:
for i in range(3, 25):
    print(i)
    condition = 1
    j = 1
    while condition == 1:
        #Fetch the HTML content of the page
        if i < 10:
            url = f'https://basketball.realgm.com/ncaa/stats/200{str(i)}/Averages/Qualified/All/Season/All/points/desc/{str(j)}'  # Replace with the URL of the site containing the table
        else:
            url = f'https://basketball.realgm.com/ncaa/stats/20{str(i)}/Averages/Qualified/All/Season/All/points/desc/{str(j)}'
        response = requests.get(url)

        #Check if the request was successful
        if response.status_code == 200:
            html_content = response.content
        else:
            print(f"Failed to retrieve the page. Status code: {response.status_code}")
            

        #Parse the HTML content using BeautifulSoup
        soup = BeautifulSoup(html_content, 'html.parser')

        try:
            #Locate the table in the HTML
            table = soup.find('table') 

            #Extract the table headers (if any)
            headers = []
            header_row = table.find('thead').find('tr') if table.find('thead') else table.find('tr')
            for th in header_row.find_all('th'):
                headers.append(th.text.strip())

            #Extract the table rows
            rows = []
            for row in table.find('tbody').find_all('tr') if table.find('tbody') else table.find_all('tr'):
                cells = row.find_all('td')
                if len(cells) > 0:  #Ensure the row contains data
                    rows.append([cell.text.strip() for cell in cells])

        except:
            if i < 10:
                df.to_csv(f'NCAA 200{str(i)} Avg.csv', index=False)
                condition += 1
                j = 1
            else:
                df.to_csv(f'NCAA 20{str(i)} Avg.csv', index=False)
                condition += 1
                j = 1
        #Convert the data into a pandas DataFrame
        if j == 1:
            df = pd.DataFrame(rows, columns=headers)
        else:
            try:
                df1 = pd.DataFrame(rows, columns=headers)
                df = pd.concat([df, df1])
            except:
                continue
        j = j + 1
    

In [3]:
list_of_stat = ['Averages', 'Advanced', 'Per_48', 'Totals', 'Misc_Stats']

In [ ]:
import pandas as pd
import requests
from bs4 import BeautifulSoup

for i in range(3, 25):  #from 2003 to 2025
    for stat in list_of_stat:
        condition = 1
        j = 1
        df = pd.DataFrame()  #reinitalise every new years

        while condition == 1:
            #Fetch the HTML content of the page
            if i < 10:
                url = f'https://basketball.realgm.com/ncaa/stats/200{str(i)}/{stat}/Qualified/All/Season/All/points/desc/{str(j)}/?pace_adjustment='
            else:
                url = f'https://basketball.realgm.com/ncaa/stats/20{str(i)}/Averages/Qualified/All/Season/All/points/desc/{str(j)}/?pace_adjustment='
            response = requests.get(url)

            if response.status_code == 200:
                html_content = response.content
            else:
                print(f"Failed to retrieve the page. Status code: {response.status_code}")
                if not df.empty:
                    df.to_csv(f'NCAA 20{str(i)} Avg.csv', index=False)
                break

            #Parse the HTML content using BeautifulSoup
            soup = BeautifulSoup(html_content, 'html.parser')

            try:
                #Locate the table in the HTML
                table = soup.find('table')
                if not table:
                    print(f"No table found on page {i}-{j}, skipping this page.")
                    j += 1
                    condition += 1
                    

                #Extract headers
                headers = [th.text.strip() for th in table.find_all('th')]  # Assuming headers are in <th> tags

                #Extract rows
                rows = []
                for row in table.find_all('tr'):
                    cells = row.find_all('td')
                    if len(cells) > 0:  #onyl rows containing data
                        rows.append([cell.text.strip() for cell in cells])
            
            except Exception as e:
                print(f"Error occurred: {e}")
                j += 1
                continue

            #Convert rows into DataFrame
            if j == 1:
                if rows:
                    df = pd.DataFrame(rows, columns=headers)  #first page
                    print(f"DataFrame created for page {i}-{j}, shape: {df.shape}")
                else:
                    print(f"No data found on page {i}-{j}, skipping.")
                    condition += 1
            else:
                if rows:
                    df1 = pd.DataFrame(rows, columns=headers)
                    print(f"Concatenating data, shape before: {df.shape}, shape of new data: {df1.shape}")
                    df = pd.concat([df, df1], ignore_index=True)
                else:
                    print(f"No data found on page {i}-{j}, skipping concatenation.")
                    condition += 1
                    print(condition)
            
            #Next page
            j += 1

        if not df.empty:
            if i < 10:
                df.to_csv(f'NCAA 200{str(i)} {stat}.csv', index=False)
            else:
                df.to_csv(f'NCAA 20{str(i)} {stat}.csv', index=False)


DataFrame created for page 3-1, shape: (200, 23)
Concatenating data, shape before: (200, 23), shape of new data: (200, 23)
Concatenating data, shape before: (400, 23), shape of new data: (200, 23)
Concatenating data, shape before: (600, 23), shape of new data: (200, 23)
Concatenating data, shape before: (800, 23), shape of new data: (200, 23)
Concatenating data, shape before: (1000, 23), shape of new data: (200, 23)
Concatenating data, shape before: (1200, 23), shape of new data: (200, 23)
Concatenating data, shape before: (1400, 23), shape of new data: (200, 23)
Concatenating data, shape before: (1600, 23), shape of new data: (200, 23)
Concatenating data, shape before: (1800, 23), shape of new data: (200, 23)
Concatenating data, shape before: (2000, 23), shape of new data: (138, 23)
No table found on page 3-12, skipping this page.
Error occurred: 'NoneType' object has no attribute 'find_all'
DataFrame created for page 3-1, shape: (200, 23)
Concatenating data, shape before: (200, 23), 

In [38]:
condition

1

In [31]:
df.to_csv('test.csv', index=False)

In [ ]:
totals, per 36, advanced

## Team stats

From https://www.sports-reference.com/cbb/seasons/men/2025-school-stats.html, I scrap the regular season information that I need, both for opponents and for the team. 

# Second part - NCAA Dataset Creation

In this second part of the Notebook, I will work on creating a functionnal Dataset for my predicting task. 
In short, my task consists at predicting future performances of collegiate players before entering into the NBA.
This will be completed by trying a bunch of variables on my use case. Here is a glimpse at the variables I will work with:
+ Past performances
    + Advanced - Totals - Averages - Per 48 - Misc (retrieved from RealGM)
+ Categorical data on a player:
    + Team - Conference - Age - Height - Weight - Role - Class (retrieved from Kaggle and Kp)
+ Exogenous factors:
    + Surrounding during the game (PBP Kuehn Valuation)
    + Team playstyle (retrieved from Kaggle)

The rest will come after...

Try to use this : https://github.com/dcstats/CBBpy

In [ ]:
#Home path to the 'College' folder
home = r"C:\Users\Utilisateur\Desktop\Master ULB\Mémoire\Database\College"

#Create a dictionnary
dico_to_access = {
    "indiv" : "Player Stats",
    "team" : "Team",
    "pace" : ["Pace adjusted 2003 to 2024 - from RealGM", "NCAA"],
    "hoopr" : ["HoopR Kp Data 2002 to 2021 - GitHub", "players_"]

}

#Choose layer for the first application
first_layer = dico_to_access["indiv"]
second_layer = dico_to_access["pace"]

## Merge the basic statistics for every player in the NCAA between 2003 to 2024

In this section, I will merge and concatenate all the Data that I have retrieved from RealGM into one single dataset. It includes:
* Averages
* Totals
* Advanced
* Miscallenaous
* Per 48

All data are pace adjusted. 

In [ ]:
#Creating a list with all the subgroups from RealGM
subgroup_data = [
    "Advanced", "Per_48", "Averages", "Totals", "Misc_Stats"
]

#Initializing an empty dataframe
df = pd.DataFrame()

#Creating a loop that will into each subgroup one-by-one and concatenates year-by-year
for sub in subgroup_data:
    #Create a subgroup df that will become empty at each subgroup iteration
    sub_df = pd.DataFrame()
    for i in range(3, 25):
        #Access the right dataframe
        if i < 10:
            season = f"200{str(i)}"
        else:
            season = f"20{str(i)}"
        path = home + rf"\{first_layer}\{second_layer[0]}\{sub}\{second_layer[1]} {season} {sub}.csv"
        current = pd.read_csv(path)
        #Change the prefix of a column except for the column Player
        current = current.add_prefix(f"{sub}_")
        current = current.rename(columns={
            f"{sub}_Player" : "Player",
            f"{sub}_Team" : "Team"
            })
        #Add the column with year (in string for extraction purpose)
        current['Season'] = season
        #if it is the first iteration, do not concatenate
        if sub_df.empty:
            sub_df = current.copy()
        else:
            sub_df = pd.concat([sub_df, current])
    if df.empty:
        df = sub_df.copy()
    else:
        df = pd.merge(df, sub_df, on=["Player", "Season", "Team"])

To this dataframe, add additional data from HoopR and Kp (GitHub). This will be done with the same looping idea and then we will merge both dataset.

In [6]:
#Initializing an empty dataframe
Kp = pd.DataFrame()
second_layer = dico_to_access["hoopr"]
#Creating a loop that will into each subgroup one-by-one and concatenates year-by-year
#Create a subgroup df that will become empty at each subgroup iteration
for i in range(2, 22):
    #Access the right dataframe
    if i < 10:
        season = f"200{str(i)}"
    else:
        season = f"20{str(i)}"
    path = home + rf"\{first_layer}\{second_layer[0]}\{second_layer[1]}{season}.csv"
    current = pd.read_csv(path)
    #Change the prefix of a column except for the column Player
    current = current.add_prefix(f"Kp_")
    current = current.rename(columns={
            f"Kp_Player" : "Player",
            f"Kp_Team" : "Team"
            })
    #Add the column with year (in string for extraction purpose)
    current['Season'] = season
    #if it is the first iteration, do not concatenate
    if Kp.empty:
        Kp = current.copy()
    else:
        Kp = pd.concat([Kp, current])

#df = pd.merge(df, Kp, on=["Player", "Season", "Team"])

Data from Kp and df cannot be merged directly, it needs a bit of cleaning.

### Cleaning the doc

In [7]:
import re
team = pd.read_excel(home + r"\Team\mbb_team_draft.xlsx", sheet_name="Sheet4")

In [8]:
team = team['market'].str.split('NCAA Division ID1')
team = team.explode('market')
team = pd.DataFrame(team, columns=["market"])

In [ ]:
def extract_info(text):
    pattern = re.compile(
        r'^(?P<nom>[A-Z][a-z\'’\.]*(?:\s[A-Z][a-z\'’\.]*)*)'  
        r'(?P<alias>[A-Z]+)'                                  
        r'(?P<surnom>[A-Z][a-z]*(?:\s[A-Z][a-z]*)*)'          
        r'(?P<NCAA_Code>\d+)'                                 
        r'(?P<Conference>.*?)'                                
        r'(?P<Conf_alias>[A-Z]+)$'                            
    )
    match = pattern.match(text)
    if match:
        return match.groups()
    return (None, None, None, None, None, None)

team[['Team', 'alias', 'surnom', 'NCAA Code', 'Conference', 'Conf alias']] = team['market'
                                                                                 ].apply(extract_info).apply(pd.Series)
#team = team.drop(columns=['market'])

team.to_excel("mbb_team.xlsx")

### Merging

Problem with the Kp dataset is that the Team column doesn't contain the abreviation of the team name.

In [32]:
team = pd.read_excel(home+r'\Team\Verif\vérification.xlsx')

In [ ]:
team['Match_bis'] = team['alias'].str.lower().isin(df['Team'].str.lower())
team['Match_bis'].value_counts()

#team.to_excel("verif3.xlsx")
#df['Team'].value_counts().to_excel("value_count_RealGM.xlsx")

In [ ]:
#Put the alias in the database
Kp = pd.merge(Kp, team, on=["Team"])
#Change the column name to be able to merge it with the other database
Kp = Kp.rename(columns= {
    'Team': 'Name',
    'alias' : 'Team'
})

In [55]:
#Merge the two databases
pd.merge(df, Kp, on=["Player", "Season"])

0           Ruben Douglas
1         Henry Domercant
2          Michael Watson
3              Mike Helms
4             Luis Flores
               ...       
40510        Robin Duncan
40511          Nobal Days
40512       Deshon Parker
40513     Gabe Osabuohien
40514    Brock Cunningham
Name: Player, Length: 40515, dtype: object

In [ ]:
#Merge the two databases
pd.merge(df, Kp, on=["Player", "Season"]).to_excel("first_look_db.xlsx")

### Conclusion

Here, we have a data set with the first two components from above : Categorical and Past data. Nothing is cleaned yet.

## PBP

In this section, I will try to replicate Kuehn's player value variable. In its research, he assumed that the value of player is a function of its surrounding: the skills and tendencies from its four teammates and its opponents. From this, he used the 73 different outcomes of basketball and calculated the probability of each outcome. 

He used PBP data to derive the player's value.

## Team playstyle